### Autograd в PyTorch

https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html

При обучении нейронных сетей мы минимизируем значение функции ошибки на обучающей выборке, меняя значения параметров модели. Чтобы понять, куда нужно сместить значения параметров, нужно уметь считать градиент — именно для автоматизации этих расчётов нам и нужен Pytorch. Посмотрим, как именно он это делает.

Но перед этим вспомним правило дифференцирования сложной функции:

$
\begin{align}
\frac{\partial f}{\partial x} =
\textcolor{teal}{\frac{\partial f}{\partial u}} \cdot 
\textcolor{violet}{\frac{\partial u}{\partial x}}
\end{align}
$

Пример:

$$
f(x) = \textcolor{teal}{\sin}(\textcolor{violet}{\ln x}) \quad
\textcolor{teal}{f(u) = \sin(u)} \quad
\textcolor{violet}{u(x) = \ln x}
$$

$$
\frac{\partial f}{\partial x} =
\textcolor{teal}{\frac{\partial f}{\partial u}} \cdot 
\textcolor{violet}{\frac{\partial u}{\partial x}} = 
\textcolor{teal}{\cos(u)} \cdot 
\textcolor{violet}{\frac{1}{x}} = 
\cos(\ln x) \cdot 
\frac{1}{x}
$$

#### 1. Дифференцирование и вычислительный граф

Рассмотрим выражение $f(x, y) = x^2 + xy + (x + y)^2$ и построим его граф:

<img src="../assets/images/forward_pass.png" style="background:white" width="300"/>

Производная по переменной $x$:
$$
\begin{align*}
\frac{\partial f}{\partial x} &=
\textcolor{violet}{
\frac{\partial f}{\partial d} \cdot 
\frac{\partial d}{\partial a} \cdot 
\frac{\partial a}{\partial x}
}
+
\textcolor{teal}{
\frac{\partial f}{\partial d} \cdot 
\frac{\partial d}{\partial b} \cdot 
\frac{\partial b}{\partial x}
}
+
\textcolor{orange}{
\frac{\partial f}{\partial e} \cdot 
\frac{\partial e}{\partial c} \cdot 
\frac{\partial c}{\partial x}
} \\
&=
\textcolor{violet}{
1 \cdot 1 \cdot 2x
}
+
\textcolor{teal}{
1 \cdot 1 \cdot y
}
+
\textcolor{orange}{
1 \cdot 2c \cdot 1
}
=
2x + y + 2(x+y)
\end{align*}
$$

Для вычисления производных Pytorch строит вычислительный граф, проход по которому позволяет рассчитать градиенты по правилу производной сложной функции (chain rule).

Прямой проход:
- расчёт значения выходного тензора
- построение графа и сохранение нужных для обратного прохода данных для каждой операции

Обратный проход (вызов `.backward()` у корня графа):
- расчёт градиентов и их накопление в атрибуте `.grad` каждого тензора
- распространение вычислений далее до листьев графа

Запишем выражение для $f(x, y)$, задав начальные условия $x = 2.0, y = 2.0$.

In [1]:
import torch

In [2]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
a = x**2
b = x * y
c = x + y
d = a + b
e = c**2
f = d + e
f

tensor(24., grad_fn=<AddBackward0>)

`grad_fn` означает, что `f` не просто отдельный тензор, а связан с вычислительным графом и соответствует операции `Add`

Запустим backprop и убедимся, что градиенты рассчитаны правильно:

$\frac{\partial f}{\partial x} = 2x + y + 2(x + y)$

$\frac{\partial f}{\partial y} = x + 2(x + y)$

In [3]:
f.backward()
print(x.grad)
print(y.grad)

tensor(14.)
tensor(10.)


#### Важно: градиенты накапливаются!

По умолчанию `.backward()` **прибавляет** новые градиенты к уже существующим в `.grad`. Если не обнулить градиенты между вызовами, они будут расти:

In [4]:
def f(x, y):
    return x**2 + x * y + (x + y)**2

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

# первый вызов
f(x, y).backward()
print("После первого backward:", x.grad, y.grad)

# второй вызов — без обнуления градиентов
f(x, y).backward()
print("После второго backward:", x.grad, y.grad)  # удвоились!

После первого backward: tensor(14.) tensor(10.)
После второго backward: tensor(28.) tensor(20.)


Поэтому при итеративной оптимизации мы будем обнулять градиенты после каждого шага, присвоив `None` в `.grad`:

In [5]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

# первый вызов
f(x, y).backward()
print("После первого backward:", x.grad, y.grad)

# обнуляем градиенты
x.grad = None
y.grad = None

# второй вызов — градиенты считаются заново
f(x, y).backward()
print("После второго backward:", x.grad, y.grad)

После первого backward: tensor(14.) tensor(10.)
После второго backward: tensor(14.) tensor(10.)


#### 2. Отключение расчёта градиентов

Иногда мы хотим локально отключить расчёт градиентов — например, при обновлении параметров или при оценке модели на тестовых данных. Для этого используется менеджер контекста `torch.no_grad()`:

In [6]:
x = torch.tensor(1.0, requires_grad=True)

y = x + 1
print(f"x.requires_grad={x.requires_grad}, y.requires_grad={y.requires_grad}")

with torch.no_grad():
    y = x + 1
print(f"x.requires_grad={x.requires_grad}, y.requires_grad={y.requires_grad}")

x.requires_grad=True, y.requires_grad=True
x.requires_grad=True, y.requires_grad=False
